In [0]:
from pyspark.sql.functions import current_timestamp

spark.sql("""
INSERT OVERWRITE delivery_cata.gold_ai.master_delivery_fact
SELECT
    d.delivery_id,
    o.order_id,
    dr.driver_id,
    dr.name                 AS driver_name,
    CAST(dr.phone AS STRING) AS driver_phone,
    dr.status               AS driver_status,
    v.vehicle_id,
    v.vehicle_type,
    v.plate_number          AS vehicle_plate_number,
    v.capacity_kg           AS vehicle_capacity_kg,
    a.area_id,
    a.area_name,
    a.city,
    CAST(a.pincode AS STRING) AS pincode,
    a.delivery_charge,
    u.user_id               AS customer_id,
    u.name                  AS customer_name,
    u.email                 AS customer_email,
    CAST(u.phone AS STRING) AS customer_phone,
    u.address               AS customer_address,
    o.order_amount,
    o.order_status,
    o.pickup_location,
    o.delivery_location,
    TO_DATE(o.created_at)   AS order_date,
    d.delivery_status,
    d.pickup_time,
    d.delivery_time,
    d.distance_km,
    p.payment_id,
    p.payment_method,
    p.payment_status,
    p.amount                AS payment_amount,
    p.payment_time,
    current_timestamp()     AS _loaded_at

FROM       delivery_cata.silver.deliveries  d

JOIN       delivery_cata.silver.orders      o  
    ON d.order_id = o.order_id 
    AND o.is_current = true

JOIN       delivery_cata.silver.users       u  
    ON o.user_id = u.user_id 
    AND u.is_current = true

JOIN       delivery_cata.silver.drivers     dr 
    ON d.driver_id = dr.driver_id  
    AND dr.is_current = true

JOIN       delivery_cata.silver.vehicles    v  
    ON dr.vehicle_id = v.vehicle_id  
    AND v.is_current = true

JOIN       delivery_cata.silver.payments    p  
    ON o.order_id = p.order_id  
    AND p.is_current = true

JOIN       delivery_cata.silver.areas       a  
    ON dr.area_id = a.area_id  
    AND a.is_current = true

WHERE d.is_current = true
""")

cnt = spark.table("delivery_cata.gold_ai.master_delivery_fact").count()
print(f"master_delivery_fact: {cnt} rows")

master_delivery_fact: 1418 rows


In [0]:
from pyspark.sql.functions import current_timestamp

#driver_performance_daily
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_ai.driver_performance_daily
SELECT
    driver_id,
    driver_name,
    vehicle_type,
    DATE(pickup_time)                                                       AS delivery_date,
    COUNT(delivery_id)                                                      AS total_orders_assigned,
    SUM(CASE WHEN delivery_status = 'DELIVERED'  THEN 1 ELSE 0 END)        AS total_delivered,
    SUM(CASE WHEN delivery_status = 'FAILED'     THEN 1 ELSE 0 END)        AS total_failed,
    SUM(CASE WHEN delivery_status = 'IN_TRANSIT' THEN 1 ELSE 0 END)        AS total_cancelled,
    ROUND(SUM(CASE WHEN delivery_status = 'DELIVERED' THEN 1 ELSE 0 END)
          * 100.0 / COUNT(delivery_id), 2)                                  AS success_rate_pct,
    ROUND(AVG((unix_timestamp(delivery_time) - unix_timestamp(pickup_time))
          / 60.0), 2)                                                       AS avg_delivery_time_mins,
    ROUND(AVG(distance_km), 2)                                              AS avg_distance_km,
    ROUND(SUM(distance_km), 2)                                              AS total_distance_km,
    COUNT(DISTINCT area_id)                                                 AS areas_covered,
    current_timestamp()                                                     AS _loaded_at
FROM delivery_cata.gold_ai.master_delivery_fact
GROUP BY driver_id, driver_name, vehicle_type, DATE(pickup_time)
""")
print(f"driver_performance_daily: {spark.table('delivery_cata.gold_ai.driver_performance_daily').count()} rows")

#area_daily_summary
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_ai.area_daily_summary
SELECT
    area_id,
    area_name,
    city,
    pincode,
    DATE(pickup_time)                                                       AS delivery_date,
    COUNT(delivery_id)                                                      AS total_orders,
    SUM(CASE WHEN delivery_status = 'DELIVERED'  THEN 1 ELSE 0 END)        AS total_delivered,
    SUM(CASE WHEN delivery_status = 'FAILED'     THEN 1 ELSE 0 END)        AS total_failed,
    SUM(CASE WHEN delivery_status = 'IN_TRANSIT' THEN 1 ELSE 0 END)        AS total_cancelled,
    ROUND(SUM(payment_amount), 2)                                           AS total_revenue,
    ROUND(AVG((unix_timestamp(delivery_time) - unix_timestamp(pickup_time))
          / 60.0), 2)                                                       AS avg_delivery_time_mins,
    ROUND(AVG(distance_km), 2)                                              AS avg_distance_km,
    COUNT(DISTINCT driver_id)                                               AS unique_drivers,
    COUNT(DISTINCT customer_id)                                             AS unique_customers,
    current_timestamp()                                                     AS _loaded_at
FROM delivery_cata.gold_ai.master_delivery_fact
GROUP BY area_id, area_name, city, pincode, DATE(pickup_time)
""")
print(f"area_daily_summary: {spark.table('delivery_cata.gold_ai.area_daily_summary').count()} rows")

#customer_order_summary
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_ai.customer_order_summary
SELECT
    customer_id,
    customer_name,
    customer_email,
    customer_phone,
    customer_address,
    COUNT(DISTINCT order_id)                                                AS total_orders,
    SUM(CASE WHEN delivery_status = 'DELIVERED'  THEN 1 ELSE 0 END)        AS total_delivered,
    SUM(CASE WHEN order_status    = 'CANCELLED'  THEN 1 ELSE 0 END)        AS total_cancelled,
    ROUND(SUM(payment_amount), 2)                                           AS total_spent,
    ROUND(AVG(payment_amount), 2)                                           AS avg_order_value,
    -- most used payment method per customer
    FIRST(payment_method)                                                   AS preferred_payment_method,
    -- most ordered area per customer
    FIRST(area_name)                                                        AS most_ordered_area,
    MIN(order_date)                                                         AS first_order_date,
    MAX(order_date)                                                         AS last_order_date,
    current_timestamp()                                                     AS _loaded_at
FROM delivery_cata.gold_ai.master_delivery_fact
GROUP BY customer_id, customer_name, customer_email,
         customer_phone, customer_address
""")
print(f"customer_order_summary: {spark.table('delivery_cata.gold_ai.customer_order_summary').count()} rows")

#payment_summary
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_ai.payment_summary
SELECT
    DATE(payment_time)                                                      AS payment_date,
    payment_method,
    area_name,
    city,
    pincode,
    COUNT(payment_id)                                                       AS total_transactions,
    ROUND(SUM(payment_amount), 2)                                           AS total_revenue,
    ROUND(AVG(payment_amount), 2)                                           AS avg_transaction_value,
    SUM(CASE WHEN payment_status = 'COMPLETED'
              OR payment_status  = 'SUCCESS'   THEN 1 ELSE 0 END)          AS completed_payments,
    SUM(CASE WHEN payment_status = 'FAILED'    THEN 1 ELSE 0 END)          AS failed_payments,
    SUM(CASE WHEN payment_status = 'PENDING'   THEN 1 ELSE 0 END)          AS pending_payments,
    current_timestamp()                                                     AS _loaded_at
FROM delivery_cata.gold_ai.master_delivery_fact
WHERE payment_method IS NOT NULL
GROUP BY DATE(payment_time), payment_method, area_name, city, pincode
""")
print(f"payment_summary: {spark.table('delivery_cata.gold_ai.payment_summary').count()} rows")

#vehicle_utilization
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_ai.vehicle_utilization
SELECT
    vehicle_id,
    vehicle_type,
    vehicle_plate_number,
    vehicle_capacity_kg,
    driver_id,
    driver_name,
    DATE(pickup_time)                                                       AS utilization_date,
    COUNT(delivery_id)                                                      AS total_deliveries,
    ROUND(SUM(distance_km), 2)                                              AS total_distance_km,
    ROUND(AVG(distance_km), 2)                                              AS avg_distance_per_trip_km,
    SUM(CASE WHEN delivery_status = 'DELIVERED'  THEN 1 ELSE 0 END)        AS total_delivered,
    SUM(CASE WHEN delivery_status = 'FAILED'     THEN 1 ELSE 0 END)        AS total_failed,
    current_timestamp()                                                     AS _loaded_at
FROM delivery_cata.gold_ai.master_delivery_fact
GROUP BY vehicle_id, vehicle_type, vehicle_plate_number,
         vehicle_capacity_kg, driver_id, driver_name, DATE(pickup_time)
""")
print(f"vehicle_utilization: {spark.table('delivery_cata.gold_ai.vehicle_utilization').count()} rows")

#Final summary
print("\n" + "-"*50)
print("GOLD LAYER COMPLETE")
print("-"*50)
for t in ["master_delivery_fact","driver_performance_daily",
          "area_daily_summary","customer_order_summary",
          "payment_summary","vehicle_utilization"]:
    cnt = spark.table(f"delivery_cata.gold_ai.{t}").count()
    print(f"  {t:<35} → {cnt:>5} rows")

driver_performance_daily: 1260 rows
area_daily_summary: 1260 rows
customer_order_summary: 987 rows
payment_summary: 1417 rows
vehicle_utilization: 1260 rows

--------------------------------------------------
GOLD LAYER COMPLETE
--------------------------------------------------
  master_delivery_fact                →  1418 rows
  driver_performance_daily            →  1260 rows
  area_daily_summary                  →  1260 rows
  customer_order_summary              →   987 rows
  payment_summary                     →  1417 rows
  vehicle_utilization                 →  1260 rows
